# Regression Models

In [1]:
# data_path = "tape/fluorescence"
# data_path = "tape/stability"
#data_path = "flip2/amylase/random_split/"
data_path = "flip2/amylase/by_mutation/"

### Model Registry

In [2]:
from protein_benchmark_models.models import ModelRegistry
print(ModelRegistry.list())

/Users/dylanelliott/workspace/protein-benchmark-models/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


['ridge_regressor', 'mlp_regressor', 'cnn_regressor']


### Train Ridge Regressor

In [3]:
import os
import pandas as pd
from protein_benchmark_models.data import OneHotSequenceDataset

# Train dataset
train_data_path = os.path.join(f".data/{data_path}/train.csv")
train_df = pd.read_csv(train_data_path)
sequences = train_df["sequence"].tolist()
targets = train_df["target"].to_numpy()
max_seq_len = train_df["sequence"].map(lambda x: len(x)).max()
train_dataset = OneHotSequenceDataset(
    sequences=sequences,
    targets=targets,
    seq_len=max_seq_len
)

# Valid dataset
val_data_path = os.path.join(f".data/{data_path}/valid.csv")
val_df = pd.read_csv(val_data_path)
sequences = val_df["sequence"].tolist()
targets = val_df["target"].to_numpy()
val_dataset = OneHotSequenceDataset(
    sequences=sequences,
    targets=targets,
    seq_len=max_seq_len
)

# Create model
model = ModelRegistry.get("ridge_regressor")(
    alpha=1.0
)

# Train, evaluate, and save
model.train(
    experiment_name=f"{data_path.replace('/', '-')}regression",
    train_data=train_dataset,
    val_data=val_dataset,
    model_path=f".models/{data_path.replace('/', '_')}ridge_regressor",
    seed=42,
    tracking=True
)

2026/03/03 22:52:05 INFO mlflow.tracking.fluent: Experiment with name 'flip2-amylase-by_mutation-regression' does not exist. Creating a new experiment.


[ridge_regressor] Train X: (2403, 9350)
[ridge_regressor] Train y: (2403,)
[ridge_regressor] Training model
[ridge_regressor] Valid RMSE: 0.0397
[ridge_regressor] Valid R2: 0.5196
[ridge_regressor] Valid SpearmanR: 0.7186
🏃 View run overjoyed-hog-324 at: http://localhost:5000/#/experiments/5/runs/fb0aa7711dd44cd4b68560e22eced8bb
🧪 View experiment at: http://localhost:5000/#/experiments/5


### Test Ridge Regressor

In [4]:
import os
import numpy as np
from protein_benchmark_models.data import OneHotSequenceDataset
from protein_benchmark_models.utils import evaluate

# Test dataset
test_data_path = os.path.join(f".data/{data_path}/test.csv")
test_df = pd.read_csv(test_data_path)
sequences = test_df["sequence"].tolist()
targets = test_df["target"].to_numpy()
test_dataset = OneHotSequenceDataset(
    sequences=sequences,
    targets=targets,
    seq_len=max_seq_len
)

# Evaluate on test set
X = np.stack([test_dataset[i]["one_hots"].numpy().flatten() for i in range(len(test_dataset))])
y = test_dataset.targets.numpy()
metrics = evaluate(model, X, y)

print(f"Test RMSE: {metrics['rmse']:.04f}")
print(f"Test R2: {metrics['r2']:.04f}")
print(f"Test SpearmanR: {metrics['spearmanr']:.04f}")

Test RMSE: 0.0428
Test R2: 0.3606
Test SpearmanR: 0.6488


### Train MLP Regressor

In [5]:
import os
import pandas as pd
from protein_benchmark_models.data import OneHotSequenceDataset

# Train dataset
train_data_path = os.path.join(f".data/{data_path}/train.csv")
train_df = pd.read_csv(train_data_path)
sequences = train_df["sequence"].tolist()
targets = train_df["target"].to_numpy()
max_seq_len = train_df["sequence"].map(lambda x: len(x)).max()
train_dataset = OneHotSequenceDataset(
    sequences=sequences,
    targets=targets,
    seq_len=max_seq_len
)

# Valid dataset
val_data_path = os.path.join(f".data/{data_path}/valid.csv")
val_df = pd.read_csv(val_data_path)
sequences = val_df["sequence"].tolist()
targets = val_df["target"].to_numpy()
val_dataset = OneHotSequenceDataset(
    sequences=sequences,
    targets=targets,
    seq_len=max_seq_len
)

# Create model
model = ModelRegistry.get("mlp_regressor")(
    layer_dims = [max_seq_len * len(train_dataset.vocab), 1024, 1],
    hidden_activation = "ReLU",
    output_activation = "Identity",
    use_bias = True,
    norm = "layer"
)

# Train, evaluate, and save
model.train(
    experiment_name=f"{data_path.replace('/', '-')}regression",
    train_data=train_dataset,
    val_data=val_dataset,
    model_path=f".models/{data_path.replace('/', '_')}mlp_regressor",
    lr=1e-4,
    weight_decay = 0.01,
    max_epochs=5,
    batch_size=256,
    val_frequency=1,
    patience=25,
    save_model="best",
    seed=42,
    tracking=True
)

Epoch: 5/5 | train_loss: 0.0619 | val_loss: 0.0629 | best_val_loss: 0.0629: 100%|██████████| 5/5 [00:11<00:00,  2.29s/it]


[mlp_regressor] Valid RMSE: 0.0638
[mlp_regressor] Valid R2: -0.2411
[mlp_regressor] Valid SpearmanR: 0.4727
🏃 View run angry-ant-958 at: http://localhost:5000/#/experiments/5/runs/bf5b5117075b4ca48f1eb1e4ff6e5d20
🧪 View experiment at: http://localhost:5000/#/experiments/5


### Test MLP Regressor

In [6]:
import os
import numpy as np
from protein_benchmark_models.data import OneHotSequenceDataset
from protein_benchmark_models.utils import evaluate

# Test dataset
test_data_path = os.path.join(f".data/{data_path}/test.csv")
test_df = pd.read_csv(test_data_path)
sequences = test_df["sequence"].tolist()
targets = test_df["target"].to_numpy()
test_dataset = OneHotSequenceDataset(
    sequences=sequences,
    targets=targets,
    seq_len=max_seq_len
)

# Evaluate on test set
X = np.stack([test_dataset[i]["one_hots"].numpy().flatten() for i in range(len(test_dataset))])
y = test_dataset.targets.numpy()
metrics = evaluate(model, X, y)

print(f"Test RMSE: {metrics['rmse']:.04f}")
print(f"Test R2: {metrics['r2']:.04f}")
print(f"Test SpearmanR: {metrics['spearmanr']:.04f}")

Test RMSE: 0.0674
Test R2: -0.5803
Test SpearmanR: 0.4417


### Train CNN Regressor

In [7]:
import os
import pandas as pd
from protein_benchmark_models.data import TokenizedSequenceDataset

# Train dataset
train_data_path = os.path.join(f".data/{data_path}/train.csv")
train_df = pd.read_csv(train_data_path)
sequences = train_df["sequence"].tolist()
targets = train_df["target"].to_numpy()
max_seq_len = train_df["sequence"].map(lambda x: len(x)).max()
train_dataset = TokenizedSequenceDataset(
    sequences=sequences,
    targets=targets,
    seq_len=max_seq_len
)

# Valid dataset
val_data_path = os.path.join(f".data/{data_path}/valid.csv")
val_df = pd.read_csv(val_data_path)
sequences = val_df["sequence"].tolist()
targets = val_df["target"].to_numpy()
val_dataset = TokenizedSequenceDataset(
    sequences=sequences,
    targets=targets,
    seq_len=max_seq_len
)

# Create model
model = ModelRegistry.get("cnn_regressor")(
    embed_dims=[len(train_dataset.vocab), 64],
    kernel_spec=[
        [3, 32, 1],
        [3, 64, 1],
        [3, 128, 2]
    ],
    seq_length=max_seq_len,
    output_dim=1,
    padding_idx=train_dataset.vocab['<PAD>'],
    hidden_activation="LeakyReLU",
    output_activation="Identity",
    use_bias=True,
    norm="layer"
)

# Train, evaluate, and save
model.train(
    experiment_name=f"{data_path.replace('/', '-')}regression",
    train_data=train_dataset,
    val_data=val_dataset,
    model_path=f".models/{data_path.replace('/', '_')}mlp_regressor",
    lr=1e-4,
    weight_decay = 0.01,
    max_epochs=5,
    batch_size=256,
    val_frequency=1,
    patience=25,
    save_model="best",
    seed=42,
    tracking=True
)

Epoch: 5/5 | train_loss: 0.1172 | val_loss: 0.0771 | best_val_loss: 0.0771: 100%|██████████| 5/5 [00:21<00:00,  4.27s/it]


[cnn_regressor] Valid RMSE: 0.0776
[cnn_regressor] Valid R2: -0.8371
[cnn_regressor] Valid SpearmanR: 0.1007
🏃 View run salty-rook-486 at: http://localhost:5000/#/experiments/5/runs/2e3b9fdec0f64e40a21f66f47125a79b
🧪 View experiment at: http://localhost:5000/#/experiments/5


### Test CNN Regressor

In [8]:
import os
import numpy as np
from protein_benchmark_models.data import TokenizedSequenceDataset
from protein_benchmark_models.utils import evaluate

# Test dataset
test_data_path = os.path.join(f".data/{data_path}/test.csv")
test_df = pd.read_csv(test_data_path)
sequences = test_df["sequence"].tolist()
targets = test_df["target"].to_numpy()
test_dataset = TokenizedSequenceDataset(
    sequences=sequences,
    targets=targets,
    seq_len=max_seq_len
)

# Evaluate on test set
X = np.stack([test_dataset[i]["tokens"].numpy() for i in range(len(test_dataset))])
y = test_dataset.targets.numpy()
metrics = evaluate(model, X, y)

print(f"Test RMSE: {metrics['rmse']:.04f}")
print(f"Test R2: {metrics['r2']:.04f}")
print(f"Test SpearmanR: {metrics['spearmanr']:.04f}")

Test RMSE: 0.0817
Test R2: -1.3246
Test SpearmanR: 0.1456
